# Diffusion Model Acceleration Techniques - Hands-On Practice

## Learning Objectives
Upon completion of this notebook, you will be able to:
- **Implement** image generation functions with timing measurements
- **Code** inference step analysis to explore speed-quality trade-offs
- **Develop** scheduler comparison methods for different sampling algorithms
- **Analyze** model architecture and parameter counts
- **Benchmark** performance across different configurations
- **Visualize** generation results and timing comparisons

## Introduction

Diffusion models work by gradually adding noise to data during training and learning to reverse this process during inference. The inference process typically requires 50-1000 denoising steps, which can be time-consuming for practical applications. In this workshop, we will:

1. **Load and use a pretrained diffusion model** for image generation
2. **Understand the denoising process** and computational bottlenecks
3. **Analyze inference time and quality trade-offs**
4. **Establish baseline performance** for future optimization techniques

### Key Concepts:
- **Forward Process**: Gradually adding Gaussian noise to destroy data structure
- **Reverse Process**: Learning to denoise and generate new samples
- **Sampling Steps**: Number of denoising iterations required for generation
- **Computational Cost**: Memory and time requirements for inference

---
**🔥 HANDS-ON PRACTICE**: This notebook contains code completion exercises marked with `# TODO:` comments. Complete the missing implementations to build diffusion model analysis tools from scratch!

## 1. Environment Setup and Dependencies

First, we install and import the necessary libraries for our diffusion model implementation.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time
from PIL import Image
from diffusers import StableDiffusionPipeline, DDIMScheduler, DDPMScheduler
import warnings
warnings.filterwarnings('ignore')

# Set device and memory optimization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    # cuDNN's SDPA backend has no execution plan for some shapes on MIG slices.
    # Disable it so torch.nn.functional.scaled_dot_product_attention falls back
    # to flash / efficient / math backends, which work on all MIG partitions.
    torch.backends.cuda.enable_cudnn_sdp(False)

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)

## 2. Loading Pretrained Diffusion Model

We'll use Stable Diffusion, a popular text-to-image diffusion model. The model consists of several components:
- **Text Encoder**: Converts text prompts to embeddings
- **UNet**: The core denoising network
- **VAE Decoder**: Converts latent representations to images
- **Scheduler**: Controls the denoising process

In [ ]:
# Load Stable Diffusion model
model_id = "runwayml/stable-diffusion-v1-5"

try:
    # Load the pipeline with memory optimization
    pipe = StableDiffusionPipeline.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        safety_checker=None,
        requires_safety_checker=False
    )
    
    # Move to device and optimize memory
    pipe = pipe.to(device)
    
    # Apply memory optimization techniques (updated for newer diffusers versions)
    if device.type == "cuda":
        # Enable memory efficient attention and slicing
        pipe.enable_attention_slicing()
        # Enable CPU offloading if needed to save GPU memory
        # pipe.enable_sequential_cpu_offload()  # Uncomment if you have memory issues
        
        # Enable xformers for better memory efficiency (if available)
        try:
            pipe.enable_xformers_memory_efficient_attention()
            print("✅ XFormers memory efficient attention enabled")
        except Exception:
            print("ℹ️  XFormers not available, using default attention")
    
    print("✅ Stable Diffusion model loaded successfully!")
    print(f"Model components:")
    print(f"  - Text Encoder: {type(pipe.text_encoder).__name__}")
    print(f"  - UNet: {type(pipe.unet).__name__}")
    print(f"  - VAE: {type(pipe.vae).__name__}")
    print(f"  - Scheduler: {type(pipe.scheduler).__name__}")
    
except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("This might be due to memory constraints or missing dependencies.")
    print("💡 Try installing xformers for better memory efficiency: pip install xformers")

## 3. Model Architecture Analysis

Let's examine the model architecture to understand the computational requirements and identify potential optimization targets.

In [ ]:
# Analyze model architecture and parameters - HANDS-ON EXERCISE

def count_parameters(model):
    """Count the total number of parameters in a model"""
    # TODO: Calculate the total number of parameters
    # HINT: Use sum() with p.numel() for each parameter in model.parameters()
    return # Your code here

def analyze_model_components():
    """Analyze each component of the diffusion model"""
    if 'pipe' not in globals():
        print("❌ Model not loaded. Please run the previous cell first.")
        return
    
    # TODO: Create a dictionary with model components
    # HINT: Include 'Text Encoder', 'UNet', 'VAE Encoder', 'VAE Decoder'
    components = {
        'Text Encoder': # Your code here,
        'UNet': # Your code here,
        'VAE Encoder': # Your code here,
        'VAE Decoder': # Your code here
    }
    
    print("🔍 Model Architecture Analysis:")
    print("=" * 50)
    
    total_params = 0
    for name, component in components.items():
        # TODO: Count parameters for each component
        # HINT: Use the count_parameters function you just defined
        params = # Your code here
        total_params += params
        print(f"{name:15}: {params:,} parameters ({params/1e6:.1f}M)")
    
    print("-" * 50)
    print(f"{'Total':15}: {total_params:,} parameters ({total_params/1e6:.1f}M)")
    
    # Memory usage estimation
    if torch.cuda.is_available():
        print(f"\n📊 Memory Usage:")
        print(f"Current GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
        print(f"Peak GPU memory: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

analyze_model_components()

## 4. Baseline Image Generation

Let's generate some images using the pretrained model to establish baseline performance metrics.

In [ ]:
def generate_image_with_timing(prompt, num_inference_steps=50, guidance_scale=7.5, seed=42):
    """Generate an image and measure the inference time - HANDS-ON EXERCISE"""
    if 'pipe' not in globals():
        print("❌ Model not loaded. Please run the model loading cell first.")
        return None, 0
    
    # TODO: Set seed for reproducibility
    # HINT: Use torch.Generator with device and manual_seed
    generator = # Your code here
    
    # TODO: Record the start time
    # HINT: Use time.time()
    start_time = # Your code here
    
    # TODO: Generate image using the pipeline
    # HINT: Use pipe() with prompt, num_inference_steps, guidance_scale, generator
    # Set height=512, width=512, and wrap in torch.no_grad()
    with torch.no_grad():
        image = # Your code here - call pipe and get .images[0]
    
    # TODO: Calculate inference time
    # HINT: Subtract start_time from current time.time()
    end_time = time.time()
    inference_time = # Your code here
    
    return image, inference_time

# Test prompts for demonstration
test_prompts = [
    "A serene landscape with mountains and a lake at sunset",
    "A cute robot playing with a cat in a garden",
    "An abstract painting with vibrant colors and geometric shapes"
]

print("🎨 Ready to generate images!")
print("Test prompts available:")
for i, prompt in enumerate(test_prompts, 1):
    print(f"{i}. {prompt}")

In [ ]:
# Generate images with different prompts and analyze performance
def run_baseline_experiments():
    """Run baseline experiments with different prompts and settings - HANDS-ON EXERCISE"""
    if 'pipe' not in globals():
        print("❌ Model not loaded. Please run the model loading cell first.")
        return
    
    results = []
    
    print("🚀 Running baseline experiments...")
    print("=" * 60)
    
    for i, prompt in enumerate(test_prompts):
        print(f"\n📝 Prompt {i+1}: {prompt}")
        print("-" * 40)
        
        # TODO: Generate image with timing
        # HINT: Call generate_image_with_timing with prompt and num_inference_steps=50
        image, inference_time = # Your code here
        
        if image is not None:
            print(f"⏱️  Generation time: {inference_time:.2f} seconds")
            
            # Memory usage if on GPU
            if torch.cuda.is_available():
                # TODO: Get peak GPU memory usage
                # HINT: Use torch.cuda.max_memory_allocated() / 1024**3
                memory_used = # Your code here
                print(f"🧠 Peak GPU memory: {memory_used:.2f} GB")
                torch.cuda.reset_peak_memory_stats()
            
            # TODO: Store results in a dictionary
            # HINT: Include 'prompt', 'inference_time', and 'image' keys
            results.append({
                'prompt': # Your code here,
                'inference_time': # Your code here,
                'image': # Your code here
            })
            
            # Display the generated image
            plt.figure(figsize=(8, 8))
            plt.imshow(image)
            plt.axis('off')
            plt.title(f"Generated Image {i+1}\nTime: {inference_time:.2f}s")
            plt.tight_layout()
            plt.show()
        else:
            print("❌ Failed to generate image")
    
    return results

# Run the baseline experiments
baseline_results = run_baseline_experiments()

## 5. Analyzing the Impact of Inference Steps

One of the key factors affecting both quality and speed is the number of denoising steps. Let's analyze this trade-off.

In [ ]:
def analyze_step_count_tradeoff():
    """Analyze the trade-off between inference steps and generation time/quality - HANDS-ON EXERCISE"""
    if 'pipe' not in globals():
        print("❌ Model not loaded. Please run the model loading cell first.")
        return
    
    step_counts = [10, 20, 30, 50, 100]
    test_prompt = "A beautiful sunset over a mountain landscape"
    
    results = {}
    images = {}
    
    print("🔬 Analyzing inference step trade-offs...")
    print("=" * 50)
    
    for steps in step_counts:
        print(f"\n🔄 Testing with {steps} inference steps...")
        
        # TODO: Generate image with different step counts
        # HINT: Call generate_image_with_timing with test_prompt, num_inference_steps=steps, seed=42
        image, inference_time = # Your code here
        
        if image is not None:
            # TODO: Store results and images
            # HINT: Use steps as the dictionary key
            results[steps] = # Your code here
            images[steps] = # Your code here
            print(f"   ⏱️  Time: {inference_time:.2f}s")
        else:
            print(f"   ❌ Failed to generate image with {steps} steps")
    
    # Create visualization
    if results:
        fig, axes = plt.subplots(2, len(step_counts), figsize=(20, 8))
        
        # Plot images in first row
        for i, steps in enumerate(step_counts):
            if steps in images:
                axes[0, i].imshow(images[steps])
                axes[0, i].set_title(f"{steps} steps\n{results[steps]:.1f}s")
                axes[0, i].axis('off')
        
        # TODO: Create timing analysis plot
        # HINT: Extract steps_list from results.keys() and times_list from results.values()
        steps_list = # Your code here
        times_list = # Your code here
        
        # Merge the subplot for timing graph
        for i in range(len(step_counts)):
            axes[1, i].remove()
        
        ax_timing = fig.add_subplot(2, 1, 2)
        # TODO: Plot the relationship between steps and timing
        # HINT: Use ax_timing.plot() with steps_list and times_list
        ax_timing.plot(# Your code here)
        ax_timing.set_xlabel('Number of Inference Steps')
        ax_timing.set_ylabel('Inference Time (seconds)')
        ax_timing.set_title('Inference Time vs Number of Steps')
        ax_timing.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        return results, images
    
    return None, None

# Run the step count analysis
step_results, step_images = analyze_step_count_tradeoff()

## 6. Scheduler Comparison

Different schedulers can significantly impact both generation quality and speed. Let's compare different sampling schedulers.

In [ ]:
def compare_schedulers():
    """Compare different schedulers for diffusion model inference - HANDS-ON EXERCISE"""
    if 'pipe' not in globals():
        print("❌ Model not loaded. Please run the model loading cell first.")
        return
    
    from diffusers import (DDIMScheduler, DDPMScheduler, LMSDiscreteScheduler, 
                          PNDMScheduler, EulerDiscreteScheduler)
    
    # TODO: Create a dictionary of schedulers to test
    # HINT: Load each scheduler using Scheduler.from_pretrained(model_id, subfolder="scheduler")
    schedulers = {
        'DDIM': # Your code here,
        'DDPM': # Your code here,
        'LMS': # Your code here,
        'PNDM': # Your code here,
        'Euler': # Your code here
    }
    
    test_prompt = "A cyberpunk cityscape at night with neon lights"
    num_steps = 25
    results = {}
    images = {}
    
    print("🔄 Comparing different schedulers...")
    print("=" * 50)
    
    for scheduler_name, scheduler in schedulers.items():
        print(f"\n🧪 Testing {scheduler_name} scheduler...")
        
        try:
            # TODO: Set the pipeline's scheduler
            # HINT: Assign scheduler to pipe.scheduler
            pipe.scheduler = # Your code here
            
            # TODO: Generate image with timing
            # HINT: Use generate_image_with_timing with test_prompt, num_steps, and seed=42
            image, inference_time = # Your code here
            
            if image is not None:
                # TODO: Store results
                # HINT: Use scheduler_name as the key
                results[scheduler_name] = # Your code here
                images[scheduler_name] = # Your code here
                print(f"   ⏱️  Time: {inference_time:.2f}s")
            else:
                print(f"   ❌ Failed with {scheduler_name}")
                
        except Exception as e:
            print(f"   ❌ Error with {scheduler_name}: {str(e)}")
    
    # Visualize results
    if results:
        fig, axes = plt.subplots(2, len(results), figsize=(4*len(results), 8))
        if len(results) == 1:
            axes = axes.reshape(-1, 1)
        
        # Display images
        for i, (scheduler_name, image) in enumerate(images.items()):
            axes[0, i].imshow(image)
            axes[0, i].set_title(f"{scheduler_name}\n{results[scheduler_name]:.1f}s")
            axes[0, i].axis('off')
        
        # Clear unused subplots in first row
        for i in range(len(results), len(axes[0])):
            axes[0, i].axis('off')
        
        # Plot timing comparison in second row
        for i in range(len(axes[1])):
            axes[1, i].remove()
        
        ax_timing = fig.add_subplot(2, 1, 2)
        # TODO: Extract scheduler names and times for plotting
        # HINT: Use list(results.keys()) and list(results.values())
        scheduler_names = # Your code here
        times = # Your code here
        
        bars = ax_timing.bar(scheduler_names, times, color='skyblue', alpha=0.7)
        ax_timing.set_ylabel('Inference Time (seconds)')
        ax_timing.set_title('Scheduler Performance Comparison')
        ax_timing.grid(True, alpha=0.3)
        
        # Add value labels on bars
        for bar, time in zip(bars, times):
            ax_timing.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                          f'{time:.1f}s', ha='center', va='bottom')
        
        plt.tight_layout()
        plt.show()
        
        return results, images
    
    return None, None

# Run scheduler comparison
scheduler_results, scheduler_images = compare_schedulers()

## 7. Performance Summary and Analysis

Let's summarize our findings and establish baseline metrics for future optimization work.

In [ ]:
def create_performance_summary():
    """Create a comprehensive performance summary of all experiments - HANDS-ON EXERCISE"""
    
    print("📊 PERFORMANCE SUMMARY")
    print("=" * 60)
    
    # Baseline results summary
    if 'baseline_results' in globals() and baseline_results:
        print("\n🎯 Baseline Image Generation:")
        # TODO: Calculate average generation time
        # HINT: Use sum() and len() on the inference_time values from baseline_results
        avg_time = # Your code here
        print(f"   Average generation time: {avg_time:.2f} seconds")
        print(f"   Total images generated: {len(baseline_results)}")
        
        for i, result in enumerate(baseline_results):
            print(f"   Image {i+1}: {result['inference_time']:.2f}s")
    
    # Step count analysis summary
    if 'step_results' in globals() and step_results:
        print("\n🔄 Inference Steps Analysis:")
        print("   Steps  | Time (s) | Speed-up")
        print("   -------|----------|----------")
        baseline_time = step_results.get(50, 0)
        for steps, time in sorted(step_results.items()):
            # TODO: Calculate speedup relative to baseline
            # HINT: Divide baseline_time by time (handle division by zero)
            speedup = # Your code here
            print(f"   {steps:6} | {time:8.2f} | {speedup:8.2f}x")
    
    # Scheduler comparison summary
    if 'scheduler_results' in globals() and scheduler_results:
        print("\n🧪 Scheduler Comparison:")
        print("   Scheduler | Time (s) | Relative Performance")
        print("   ----------|----------|---------------------")
        # TODO: Find the minimum time across all schedulers
        # HINT: Use min() on scheduler_results.values()
        min_time = # Your code here
        for scheduler, time in sorted(scheduler_results.items(), key=lambda x: x[1]):
            # TODO: Calculate relative performance
            # HINT: Divide time by min_time
            relative = # Your code here
            print(f"   {scheduler:9} | {time:8.2f} | {relative:8.2f}x slower")
    
    # Model architecture summary
    if 'pipe' in globals():
        print("\n🏗️  Model Architecture:")
        # TODO: Calculate total parameters across all components
        # HINT: Use count_parameters on text_encoder, unet, and vae
        total_params = # Your code here
        print(f"   Total parameters: {total_params:,} ({total_params/1e6:.1f}M)")
        
        if torch.cuda.is_available():
            print(f"   GPU memory usage: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")
    
    # Key insights
    print("\n💡 Key Insights:")
    print("   • Reducing inference steps provides significant speedup but may affect quality")
    print("   • Different schedulers offer varying speed-quality trade-offs")
    print("   • Memory optimization techniques are crucial for GPU deployment")
    print("   • Sequential nature of denoising limits parallelization opportunities")
    
    print("\n🔮 Future Optimization Opportunities:")
    print("   • Model quantization and pruning")
    print("   • Knowledge distillation to smaller models")
    print("   • Architectural optimizations (cached attention, etc.)")
    print("   • Advanced sampling techniques (DDIM, DPM-Solver)")
    print("   • Hardware-specific optimizations")

# Generate the performance summary
create_performance_summary()